# Detecção e agrupamento de tráfego com CIC-UNSW-NB15 e CIC-IDS2017

## Objetivo

Treinar modelos de classificação **normal × possível ataque** usando duas bases, comparar treino individual e conjunto e avaliar uma referência de agrupamento no mesmo teste. O modelo estima risco de ataque; não identifica com confiabilidade ataques inéditos, nem é um sistema pronto para uso operacional.

**Antes de executar:** este notebook baixa espelhos públicos de terceiros (versões fixadas abaixo); confira os termos, a integridade e a correspondência com os arquivos oficiais. Arquivos também podem ser fornecidos no Drive. As tabelas e resultados são gerados ao executar; não há métricas simuladas ou promessas de desempenho. A amostragem limitada é para viabilizar o Colab e afeta a representação e prevalência dos dados.

## Plano

1. Obter dados e conferir rótulos; 2. comparar características; 3. dividir por grupos de linhas idênticas; 4. treinar classificadores e baseline de cluster; 5. selecionar limite na validação; 6. avaliar testes por fonte, categoria e alertas; 7. explicar e salvar artefatos.

A base CIC-UNSW é derivada das capturas do UNSW original. Não constitui teste externo independente do notebook anterior. Esta experiência usa classificação supervisionada como extensão da segmentação; o baseline de clusters continua não supervisionado na formação dos grupos.

## Resultado de uma execução de referência — para conferência

O ciclo completo deste notebook foi executado com os arquivos dos espelhos fixados: amostra de 200.000 linhas do CIC-UNSW e até 30.000 linhas de cada um dos oito arquivos do CIC-IDS2017. Houve divisão por grupos de características idênticas. A Random Forest foi escolhida no conjunto de validação e avaliada no teste reservado de **65.185 conexões**.

| Estratégia | Ataques no teste | Ataques reportados | Ataques não reportados | Normais reportados | Precisão | Recall |
|---|---:|---:|---:|---:|---:|---:|
| Random Forest conjunta | 12.272 | 11.466 | 806 | 591 | 95,10% | 93,43% |
| Cluster reconstruído | 12.272 | 1.789 | 10.483 | 197 | 90,08% | 14,58% |
| Revisar tudo | 12.272 | 12.272 | 0 | 52.913 | 18,83% | 100% |

Os valores são de uma execução específica, com amostragem limitada por arquivo, limiar escolhido na validação e divisão por características comuns. Eles não garantem desempenho em rede real, não identificam o tipo do ataque e não constituem comparação direta com o notebook antigo. O teste não é separado por tempo ou sessão e as bases foram geradas em ambientes controlados. A verificação dos significados e unidades das 17 colunas comuns ainda é necessária. Nova execução pode produzir resultados distintos com versões ou arquivos diferentes.

Os resultados por fonte foram: CIC-IDS2017, recall 90,16% e precisão 98,14%; CIC-UNSW, recall 96,98% e precisão 92,22%. Esses números pertencem à amostra selecionada, cuja composição pode não representar tráfego real.


## 1. Preparar o Colab e dependências

Execute esta célula primeiro. Se optar por arquivos próprios, edite os caminhos na célula seguinte. Os downloads acessam serviços externos a partir do Colab; poderão exigir conectividade, armazenamento e tempo. O código limita a amostra por arquivo CIC, o que pode alterar a proporção entre cenários.

In [ ]:
%pip -q install "huggingface_hub>=0.22" pyarrow joblib scikit-learn pandas numpy matplotlib

In [ ]:
from pathlib import Path
import json, re, time, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report, precision_score,
    recall_score, f1_score, average_precision_score, precision_recall_curve)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
import joblib

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)
# None: baixar espelhos; para arquivos próprios, substitua pelos caminhos absolutos.
ARQUIVO_UNSW_DADOS = None    # ex.: '/content/drive/MyDrive/dados/Data.csv'
ARQUIVO_UNSW_ROTULOS = None  # ex.: '/content/drive/MyDrive/dados/Label.csv' ou 'Lable.csv'
PASTA_CIC = None             # pasta com arquivos CSV ou parquet do CIC-IDS2017
AMOSTRA_UNSW = 200_000       # amostragem aleatória simples antes da divisão; None para todos
AMOSTRA_CIC_POR_ARQUIVO = 30_000  # por arquivo, não preserva a proporção entre arquivos
SAIDA = Path('/content/resultados_modelo_multibase')
SAIDA.mkdir(parents=True, exist_ok=True)
print('Pasta de resultados:', SAIDA)

## 2. Obter e carregar as fontes

Fontes originais: [CIC-UNSW-NB15](https://www.unb.ca/cic/datasets/cic-unsw-nb15.html) e [CIC-IDS2017](https://www.unb.ca/cic/datasets/ids-2017.html). Espelhos usados aqui: [bencorn/CIC-UNSW-NB15](https://huggingface.co/datasets/bencorn/CIC-UNSW-NB15) e [bvsam/cic-ids-2017](https://huggingface.co/datasets/bvsam/cic-ids-2017). O primeiro disponibiliza `Data.csv` e `Lable.csv` separados; a posição das linhas é assumida como correspondência e conferimos o tamanho, mas isso não verifica por si só a integridade do espelho. Compare hashes/ordem com a fonte antes de resultados definitivos.

O mapeamento de rótulos numéricos segue o `Readme.txt` do espelho: 0 = Benign e 1–9 = categorias de ataque. Valores fora desse conjunto interrompem a execução. No CIC-IDS2017, os rótulos de texto originais são mantidos como categorias, e `BENIGN` indica normal.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

# As revisões fixam os espelhos que foram identificados durante a preparação.
REV_UNSW = '8b5e371'
REV_CIC = '036f984'
if ARQUIVO_UNSW_DADOS is None:
    ARQUIVO_UNSW_DADOS = hf_hub_download('bencorn/CIC-UNSW-NB15', 'Data.csv',
                                          repo_type='dataset', revision=REV_UNSW)
if ARQUIVO_UNSW_ROTULOS is None:
    ARQUIVO_UNSW_ROTULOS = hf_hub_download('bencorn/CIC-UNSW-NB15', 'Lable.csv',
                                            repo_type='dataset', revision=REV_UNSW)
if PASTA_CIC is None:
    arquivos_repo = list_repo_files('bvsam/cic-ids-2017', repo_type='dataset', revision=REV_CIC)
    paths_cic = [hf_hub_download('bvsam/cic-ids-2017', f, repo_type='dataset', revision=REV_CIC)
                 for f in sorted(arquivos_repo)
                 if f.startswith('machine_learning/') and f.endswith('.parquet')]
else:
    paths_cic = sorted([*Path(PASTA_CIC).glob('*.csv'), *Path(PASTA_CIC).glob('*.parquet')])
if not paths_cic:
    raise FileNotFoundError('Nenhum arquivo CIC encontrado; confira PASTA_CIC ou o acesso ao espelho.')
print('UNSW dados:', ARQUIVO_UNSW_DADOS)
print('UNSW rótulos:', ARQUIVO_UNSW_ROTULOS)
print('Arquivos CIC:', len(paths_cic))

In [ ]:
# Normalizar nomes sem presumir que nomes diferentes tenham o mesmo significado.
def normalizar_nome(nome):
    return re.sub(r'[^a-z0-9]+', '_', str(nome).strip().lower()).strip('_')
def normalizar_colunas(tabela):
    nomes = [normalizar_nome(x) for x in tabela.columns]
    if len(nomes) != len(set(nomes)):
        raise ValueError('Colunas duplicadas após normalização: revise os cabeçalhos.')
    tabela = tabela.copy(); tabela.columns = nomes
    # Aliases de cabeçalhos observados nas duas versões dos CSVs.
    aliases = {'total_fwd_packet':'total_fwd_packets',
        'total_bwd_packets':'total_backward_packets',
        'total_length_of_fwd_packet':'total_length_of_fwd_packets',
        'total_length_of_bwd_packet':'total_length_of_bwd_packets',
        'packet_length_min':'min_packet_length', 'packet_length_max':'max_packet_length'}
    for antigo, novo in aliases.items():
        if antigo in tabela.columns:
            if novo in tabela.columns: raise ValueError(f'Ambiguidade entre {antigo} e {novo}')
            tabela = tabela.rename(columns={antigo:novo})
    return tabela

def carregar_rotulos(path):
    rot = pd.read_csv(path, header=None)
    if rot.shape[1] != 1:
        raise ValueError('O arquivo de rótulos deve ter exatamente uma coluna; revise o arquivo.')
    vals = pd.to_numeric(rot.iloc[:, 0], errors='coerce')
    if pd.isna(vals.iloc[0]) and vals.iloc[1:].notna().all():
        vals = vals.iloc[1:]  # apenas se a primeira linha for um cabeçalho textual
    if vals.isna().any() or not (vals % 1 == 0).all():
        raise ValueError('Rótulos ausentes, não numéricos ou não inteiros.')
    vals = vals.astype(int).reset_index(drop=True)
    if not set(vals.unique()).issubset(set(range(10))):
        raise ValueError('Rótulos fora de 0–9; confira Readme.txt da fonte.')
    return vals

mapa_unsw = {0:'Benign', 1:'Analysis', 2:'Backdoor', 3:'DoS', 4:'Exploits',
             5:'Fuzzers', 6:'Generic', 7:'Reconnaissance', 8:'Shellcode', 9:'Worms'}
unsw = normalizar_colunas(pd.read_csv(ARQUIVO_UNSW_DADOS, low_memory=False))
rotulos = carregar_rotulos(ARQUIVO_UNSW_ROTULOS)
if len(unsw) != len(rotulos):
    raise ValueError(f'Linhas incompatíveis: dados={len(unsw)}, rótulos={len(rotulos)}. Confira cabeçalho e origem.')
unsw['categoria_real'] = rotulos.map(mapa_unsw).to_numpy()
unsw['ataque'] = (rotulos != 0).astype(int).to_numpy()
unsw['origem'] = 'CIC-UNSW'
print('Distribuição original UNSW:'); display(unsw['categoria_real'].value_counts())
if AMOSTRA_UNSW and len(unsw) > AMOSTRA_UNSW:
    unsw = unsw.sample(n=AMOSTRA_UNSW, random_state=SEMENTE).reset_index(drop=True)

partes_cic = []
for i, caminho in enumerate(paths_cic):
    parte = pd.read_parquet(caminho) if str(caminho).lower().endswith('.parquet') else pd.read_csv(caminho, low_memory=False)
    parte = normalizar_colunas(parte)
    if 'label' not in parte.columns:
        raise ValueError(f'Coluna Label não encontrada em {caminho}. Confira o arquivo.')
    if parte['label'].isna().any():
        raise ValueError(f'Rótulos ausentes em {caminho}; investigue antes de continuar.')
    parte['categoria_real'] = parte['label'].astype(str).str.strip()
    if parte['categoria_real'].str.fullmatch(r'\d+(\.0)?').any():
        raise ValueError('Rótulos numéricos no CIC-IDS2017 exigem mapeamento verificado; não adivinhe.')
    parte['ataque'] = (~parte['categoria_real'].str.lower().isin(['benign', 'normal'])).astype(int)
    parte['origem'] = 'CIC-IDS2017'
    n_original = len(parte)
    if AMOSTRA_CIC_POR_ARQUIVO and n_original > AMOSTRA_CIC_POR_ARQUIVO:
        parte = parte.sample(n=AMOSTRA_CIC_POR_ARQUIVO, random_state=SEMENTE+i)
    print(Path(caminho).name, ':', n_original, '→', len(parte), 'linhas')
    partes_cic.append(parte)
cic = pd.concat(partes_cic, ignore_index=True)
print('Amostra UNSW:', unsw.shape, '| amostra CIC:', cic.shape)
print('Distribuição CIC na amostra:'); display(cic['categoria_real'].value_counts())

## 3. Compatibilidade de características e EDA

Utilizamos uma lista conservadora de nomes idênticos após normalização e aliases explícitos para nomes de contagem e tamanho com grafias observadas, relativos a duração, quantidade de pacotes, tamanhos e intervalos. Não usamos IP, porta, timestamp, identificador, rótulo ou origem como característica. Mesmo com o mesmo nome, a semântica e unidades podem diferir entre versões do CICFlowMeter: **esta checagem não substitui a conferência documental das colunas e configurações**. Se menos de quatro colunas da lista estiverem disponíveis nas duas fontes, a execução para para evitar inventar equivalências.

A amostragem por arquivo altera a contribuição relativa dos dias e pode deixar categorias raras pouco representadas. Ela não representa o tráfego de uma rede real.

In [ ]:
candidatas = ['flow_duration', 'total_fwd_packets', 'total_backward_packets',
 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_mean',
 'bwd_packet_length_mean', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean',
 'fwd_iat_mean', 'bwd_iat_mean', 'min_packet_length', 'max_packet_length',
 'packet_length_mean', 'packet_length_std', 'packet_length_variance']
comuns = [x for x in candidatas if x in unsw.columns and x in cic.columns]
print('Características comuns escolhidas:', comuns)
print('Faltantes em UNSW:', [x for x in candidatas if x not in unsw.columns])
print('Faltantes em CIC:', [x for x in candidatas if x not in cic.columns])
if len(comuns) < 4:
    raise ValueError('Menos de 4 características reconhecidas em comum. Inspecione cabeçalhos e verifique unidades; não renomeie por suposição.')
colunas = comuns + ['origem', 'categoria_real', 'ataque']
dados = pd.concat([unsw[colunas], cic[colunas]], ignore_index=True)
for coluna in comuns:
    dados[coluna] = pd.to_numeric(dados[coluna], errors='coerce')
dados[comuns] = dados[comuns].replace([np.inf, -np.inf], np.nan)
print('Distribuição por fonte e classe na amostra:')
display(pd.crosstab(dados['origem'], dados['ataque']))
print('Nulos e infinitos por característica após conversão:')
display(dados[comuns].isna().sum().sort_values(ascending=False))
display(dados[comuns].describe(percentiles=[.25, .5, .75]).T)
del unsw, cic, partes_cic

## 4. Divisão por registros com características iguais

Agrupamos linhas com valores idênticos nas características comuns e mantemos cada grupo em apenas uma partição, inclusive entre fontes. É uma proteção contra cópias exatas; **não identifica todas as sessões relacionadas nem garante independência temporal**. Agrupamentos podem produzir proporções diferentes das originais. São verificadas as duas classes em todas as partições e mostrados os tamanhos por fonte.

A seleção e os limites usam apenas validação; teste é reservado até o final. Rótulos conflitantes para características idênticas são contados, não apagados automaticamente.

In [ ]:
hashes = pd.util.hash_pandas_object(dados[comuns], index=False).to_numpy()
conflitos = dados.assign(_grupo=hashes).groupby('_grupo')['ataque'].nunique()
print('Grupos de características com rótulos conflitantes:', int((conflitos > 1).sum()))
indices = np.arange(len(dados))
split_1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEMENTE)
itr, iresto = next(split_1.split(indices, groups=hashes))
split_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEMENTE+1)
iv_local, it_local = next(split_2.split(iresto, groups=hashes[iresto]))
iv, it = iresto[iv_local], iresto[it_local]
assert not (set(hashes[itr]) & set(hashes[iv]))
assert not (set(hashes[itr]) & set(hashes[it]))
assert not (set(hashes[iv]) & set(hashes[it]))
for nome, idx in [('treino',itr), ('validação',iv), ('teste',it)]:
    print('\n', nome, ':', len(idx))
    display(pd.crosstab(dados.iloc[idx]['origem'], dados.iloc[idx]['ataque']))
    if dados.iloc[idx]['ataque'].nunique() != 2:
        raise ValueError(f'{nome} não contém as duas classes; ajuste amostragem ou divisão.')
Xtr, Xv, Xt = (dados.iloc[i][comuns] for i in (itr, iv, it))
ytr, yv, yt = (dados.iloc[i]['ataque'].to_numpy() for i in (itr, iv, it))

## 5. Baselines e novos modelos

Treinamos uma referência que sinaliza tudo, uma regra de agrupamento semelhante à anterior e classificadores supervisionados. A referência de clusters é uma **reconstrução com as características comuns**, não o modelo antigo exato. Ela faz o agrupamento sem rótulos e usa os rótulos do treino para escolher um cluster prioritário.

Os dois classificadores usam imputação e padronização ajustadas no treino. A Random Forest testa relações não lineares; a regressão logística oferece uma referência linear. Ambos são treinados com dados das duas bases. Não existe garantia de que superarão a referência.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# Modelo de agrupamento: pipeline treinado apenas no conjunto de treino.
modelo_cluster = Pipeline([
    ('imputar', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('escalar', StandardScaler()),
    ('cluster', MiniBatchKMeans(n_clusters=5, random_state=SEMENTE, batch_size=2048, n_init=5))
])
inicio = time.perf_counter()
grupo_tr = modelo_cluster.fit_predict(Xtr)
perfil = pd.DataFrame({'grupo':grupo_tr, 'ataque':ytr}).groupby('grupo')['ataque'].agg(['size','mean'])
grupo_escolhido = perfil['mean'].idxmax()
print('Grupo prioritário, escolhido no treino:', grupo_escolhido)
display(perfil)
modelos = {
 'Regressão logística': Pipeline([
    ('imputar', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('escalar', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=500, class_weight='balanced', random_state=SEMENTE))]),
 'Random Forest': Pipeline([
    ('imputar', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('modelo', RandomForestClassifier(n_estimators=100, max_depth=18, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEMENTE))])
}
tempos = {}
for nome, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(Xtr, ytr)
    tempos[nome] = time.perf_counter()-inicio
    print(f'{nome}: {tempos[nome]:.1f} segundos')

## 6. Escolha do modelo e limiar somente na validação

Para cada modelo, escolhemos na validação o limiar que maximiza o F1 binário, depois escolhemos o modelo pelo maior F1 na mesma validação. É um critério ilustrativo: na prática deve considerar custo dos falsos negativos, falsos positivos e capacidade de revisão. O resultado ainda não é uso operacional. A validação pode superajustar essa escolha; a avaliação final permanece reservada.

In [ ]:
def escolher_limiar_f1(y, scores):
    p, r, lim = precision_recall_curve(y, scores)
    if len(lim) == 0: raise ValueError('Sem limiares válidos.')
    f = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    j = int(np.argmax(f))
    return float(lim[j]), float(f[j])
linhas_validacao = []
for nome, modelo in modelos.items():
    score = modelo.predict_proba(Xv)[:, 1]
    limiar, f1 = escolher_limiar_f1(yv, score)
    linhas_validacao.append({'modelo': nome, 'limiar': limiar, 'f1_validacao': f1,
                              'average_precision': average_precision_score(yv, score),
                              'segundos_treino': tempos[nome]})
validacao = pd.DataFrame(linhas_validacao).sort_values('f1_validacao', ascending=False).reset_index(drop=True)
display(validacao.round(4))
escolhido = validacao.loc[0, 'modelo']
limiar_final = float(validacao.loc[0, 'limiar'])
modelo_final = modelos[escolhido]
print('Modelo escolhido:', escolhido, '| limiar:', round(limiar_final,4))

## 7. Avaliação final: mesmo teste para modelos e referência

Amostras de teste são as mesmas para o modelo supervisionado e o cluster reconstruído. A regra “revisar tudo” dá recall de 100%, mas encaminha todas as conexões. Tabela inclui resultados por fonte: é possível que o conjunto tenha desempenho bom em uma fonte e ruim na outra.

O resultado histórico do notebook anterior (17/23.869 ataques) é mostrado apenas como contexto; não é comparação direta porque usava outras características e outra divisão.

In [ ]:
scores_teste = modelo_final.predict_proba(Xt)[:, 1]
pred_teste = (scores_teste >= limiar_final).astype(int)
pred_cluster = (modelo_cluster.predict(Xt) == grupo_escolhido).astype(int)
meta_teste = dados.iloc[it][['origem','categoria_real']].reset_index(drop=True)

def medidas(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {'n':len(y), 'ataques':int(tp+fn), 'revisoes':int(tp+fp),
            'tp':int(tp), 'fp':int(fp), 'fn':int(fn), 'tn':int(tn),
            'recall':recall_score(y,pred,zero_division=0),
            'precisao':precision_score(y,pred,zero_division=0),
            'f1':f1_score(y,pred,zero_division=0),
            'taxa_alertas':float(np.mean(pred)),
            'taxa_falsos_positivos':float(fp/(fp+tn)) if fp+tn else np.nan}
registros = []
for fonte in ['TODAS'] + sorted(meta_teste['origem'].unique().tolist()):
    mask = np.ones(len(yt), dtype=bool) if fonte == 'TODAS' else meta_teste['origem'].eq(fonte).to_numpy()
    if not mask.any(): continue
    for nome, pred in [('Revisar tudo',np.ones(len(yt), dtype=int)),
                       ('Cluster reconstruído',pred_cluster), (escolhido,pred_teste)]:
        registros.append({'fonte':fonte, 'estrategia':nome, **medidas(yt[mask],pred[mask])})
resultado_teste = pd.DataFrame(registros)
display(resultado_teste.round(4))
resultado_teste.to_csv(SAIDA/'metricas_por_fonte.csv',index=False)
print('Histórico anterior: 17 ataques reportados de 23.869; outra divisão e outras características.')

In [ ]:
matriz = confusion_matrix(yt, pred_teste, labels=[0,1])
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(matriz, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j,i,f'{matriz[i,j]:,}'.replace(',','.'), ha='center',va='center',
                color='white' if matriz[i,j] > matriz.max()/2 else 'black',fontsize=15)
ax.set_xticks([0,1],['Não alertar','Alertar']);ax.set_yticks([0,1],['Normal','Ataque'])
ax.set_xlabel('Decisão');ax.set_ylabel('Rótulo conhecido');ax.set_title(f'Matriz de confusão — {escolhido}')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()

comp = resultado_teste[resultado_teste['fonte']=='TODAS'].set_index('estrategia')
comp[['recall','precisao','taxa_alertas']].plot.bar(figsize=(11,5), ylim=(0,1.05), rot=0)
plt.title('Recall, precisão e proporção de conexões revisadas');plt.ylabel('Proporção');plt.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()

## 8. Cobertura por categoria e exemplos

As categorias abaixo são **rótulos conhecidos de avaliação**, não categorias previstas pelo modelo binário. Nenhuma categoria desconhecida é automaticamente identificada. Poucos exemplos implicam estimativas instáveis; as bases usam taxonomias diferentes, portanto as categorias são apresentadas por fonte.

In [ ]:
diag = meta_teste.copy()
diag['ataque'] = yt; diag['alerta'] = pred_teste; diag['score'] = scores_teste
ataques = diag.loc[diag['ataque']==1]
por_tipo = ataques.groupby(['origem','categoria_real'], observed=True).agg(
    ataques=('ataque','size'), detectados=('alerta','sum'))
por_tipo['nao_detectados'] = por_tipo['ataques']-por_tipo['detectados']
por_tipo['recall'] = por_tipo['detectados']/por_tipo['ataques']
display(por_tipo.sort_values(['origem','ataques'],ascending=[True,False]).round(4))
por_tipo.to_csv(SAIDA/'cobertura_por_categoria.csv')
exemplos = dados.iloc[it][comuns].reset_index(drop=True).copy()
exemplos['origem'] = meta_teste['origem']; exemplos['categoria_referencia'] = meta_teste['categoria_real']
exemplos['score_ataque'] = scores_teste; exemplos['decisao'] = np.where(pred_teste==1,'Possível ataque — revisar','Não priorizado')
print('Exemplos com rótulo conhecido para conferência:')
display(exemplos.head(10).round(4))

## 9. Explicabilidade e análise de fontes

Os coeficientes da regressão logística e a importância interna da Random Forest são descrições do modelo, não relações causais. Como os dados foram coletados em ambientes diferentes, características podem refletir peculiaridades da fonte. Verifique também se algum atributo representa indiretamente o ambiente de coleta.

A seção 8 compara categorias, mas não mede justiça demográfica; atributos sensíveis relevantes não foram fornecidos. O resultado deve ser apresentado junto às contagens e à alteração da distribuição causada pela amostragem.

In [ ]:
etapa_modelo = modelo_final.named_steps['modelo']
if hasattr(etapa_modelo, 'feature_importances_'):
    importancia = etapa_modelo.feature_importances_
    rotulo_importancia = 'Importância interna da Random Forest'
else:
    importancia = etapa_modelo.coef_[0]
    rotulo_importancia = 'Coeficiente da regressão logística (variáveis padronizadas)'
explicacao = pd.DataFrame({'caracteristica':comuns, 'valor':importancia})
explicacao['magnitude'] = explicacao['valor'].abs()
explicacao = explicacao.sort_values('magnitude',ascending=False)
print(rotulo_importancia); display(explicacao)
explicacao.to_csv(SAIDA/'explicabilidade.csv',index=False)

## 10. Exportação, segurança e próximos passos

O arquivo exportado contém o pipeline, as colunas e o limiar escolhidos. Somente carregue arquivos joblib de fontes confiáveis: sua desserialização pode executar código. É um artefato de estudo, não um detector implantável.

Para identificar o tipo de ataque com um modelo próprio, será preciso definir uma taxonomia comum e treinar/avaliar uma tarefa multiclasse separada. Para alertas reais, obtenha dados autorizados, valide o instante de disponibilidade das características, teste outra rede e estabeleça revisão humana. O trabalho exige citar os autores das bases e a ferramenta de IA usada.

In [ ]:
artefato = {'pipeline':modelo_final, 'colunas':comuns, 'limiar':limiar_final,
    'modelo':escolhido, 'semente':SEMENTE, 'origens':['CIC-UNSW','CIC-IDS2017'],
    'aviso':'Somente pesquisa; score não necessariamente calibrado; não usar para bloqueio automático.'}
joblib.dump(artefato, SAIDA/'modelo_binario_pesquisa.joblib')
with open(SAIDA/'configuracao.json','w',encoding='utf-8') as f:
    json.dump({k:v for k,v in artefato.items() if k != 'pipeline'},f,ensure_ascii=False,indent=2)
print('Arquivos salvos em:', SAIDA)
print('No Colab, copie a pasta para seu Drive ou baixe os arquivos para não perdê-los ao encerrar a sessão.')

## 11. Treinamento conjunto comparado ao treinamento por fonte

Para medir a contribuição da segunda base, usamos o mesmo tipo de classificador escolhido anteriormente e treinamos duas cópias: uma apenas com CIC-UNSW e outra apenas com CIC-IDS2017. Cada cópia escolhe seu limiar no mesmo conjunto de validação conjunto. O resultado de cada modelo é apresentado em cada fonte do mesmo teste reservado. Isso é uma comparação de uso de fontes, não evidência de generalização para uma terceira rede. A amostragem por arquivo e os tamanhos diferentes podem favorecer uma fonte.

In [ ]:
comparacao_fontes = []
fontes_tr = dados.iloc[itr]['origem'].to_numpy()
for fonte_treino in ['CIC-UNSW', 'CIC-IDS2017']:
    mascara_tr = fontes_tr == fonte_treino
    if len(np.unique(ytr[mascara_tr])) < 2:
        print('Sem duas classes no treino de', fonte_treino, '; experimento ignorado.')
        continue
    novo_modelo = clone(modelo_final)
    novo_modelo.fit(Xtr.iloc[mascara_tr], ytr[mascara_tr])
    limiar_individual, _ = escolher_limiar_f1(yv, novo_modelo.predict_proba(Xv)[:,1])
    pred_individual = (novo_modelo.predict_proba(Xt)[:,1] >= limiar_individual).astype(int)
    for fonte_teste in sorted(meta_teste['origem'].unique()):
        mask = meta_teste['origem'].eq(fonte_teste).to_numpy()
        comparacao_fontes.append({'treino':fonte_treino, 'teste':fonte_teste,
            'limiar':limiar_individual, **medidas(yt[mask], pred_individual[mask])})
for fonte_teste in sorted(meta_teste['origem'].unique()):
    mask = meta_teste['origem'].eq(fonte_teste).to_numpy()
    comparacao_fontes.append({'treino':'Conjunto', 'teste':fonte_teste,
        'limiar':limiar_final, **medidas(yt[mask], pred_teste[mask])})
comparacao_fontes = pd.DataFrame(comparacao_fontes)
display(comparacao_fontes.round(4))
comparacao_fontes.to_csv(SAIDA/'comparacao_treino_por_fonte.csv',index=False)

## Referências

- CIC-UNSW-NB15 e método de geração: https://www.unb.ca/cic/datasets/cic-unsw-nb15.html
- CIC-IDS2017 e artigo de Sharafaldin, Lashkari e Ghorbani: https://www.unb.ca/cic/datasets/ids-2017.html
- UNSW-NB15 original e trabalhos de Moustafa e Slay: https://research.unsw.edu.au/projects/unsw-nb15-dataset
- Espelhos utilizados: https://huggingface.co/datasets/bencorn/CIC-UNSW-NB15 e https://huggingface.co/datasets/bvsam/cic-ids-2017
- Rótulos do primeiro espelho: https://huggingface.co/datasets/bencorn/CIC-UNSW-NB15/blob/main/Readme.txt
- scikit-learn, vazamento: https://scikit-learn.org/stable/common_pitfalls.html

**Crédito:** notebook preparado com apoio de ChatGPT. Revise e explique todas as decisões antes da entrega.